# ODE Solutions to SPK Files

This tutorial shows how to convert ODE solution segments into files that can be used with SPICE-based analysis and visualization workflows.

The main helper is `ode_sol_to_spk`. It writes a BSP/SPK file from one or more ODE solution segments using native SPICE SPK writing routines. It can also write maneuver and metadata sidecar files.

The current implementation writes SPK type-13 segments directly through SPICE. No external SPK-writing executable is required.

## Running this notebook

The next cell activates the environment shipped with the examples (`examples/Project.toml`), which uses the local copy of `HighFidelityEphemerisModel.jl`. Skip it if you would rather use your own environment with `HighFidelityEphemerisModel`, `OrdinaryDiffEq`, and `SPICE` installed.

SPICE kernels are loaded from the directory given by the `SPICE` environment variable, which should contain the `lsk/`, `spk/`, `pck/`, and `fk/` subfolders of the [NAIF generic kernels](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/). Edit `spice_dir` below if your kernels live elsewhere.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

## Setup

We first load the SPICE kernels and define the dynamics. Here, we consider a spacecraft in a 3000 km radius orbit around the Moon, perturbed by the Earth and the Sun.

In [ ]:
using LinearAlgebra
using OrdinaryDiffEq
using SPICE
using HighFidelityEphemerisModel

# load SPICE kernels
spice_dir = ENV["SPICE"]
furnsh(joinpath(spice_dir, "lsk", "naif0012.tls"))
furnsh(joinpath(spice_dir, "spk", "de440.bsp"))
furnsh(joinpath(spice_dir, "pck", "gm_de440.tpc"))

naif_ids = ["301", "399", "10"]        # Moon-centered, with Earth & Sun as third bodies
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 3000.0
et0 = str2et("2026-01-05T00:00:00")
parameters = SpiceParameters(et0, DU, GMs, naif_ids, "J2000", "NONE")

## Propagating coast arcs

We build a trajectory made of four 12-hour coast arcs. Impulsive maneuvers are applied at the start of the second and fourth arcs, so the velocity is discontinuous at these two boundaries. The second and third arcs are joined continuously (i.e. there is no maneuver between them).

In [ ]:
inc = deg2rad(60.0)
x0 = [1.0, 0.0, 0.0, 0.0, cos(inc), sin(inc)]   # initial state, in canonical units
arc_duration = 12 * 3600 / parameters.TU        # duration of each arc, in canonical time

# impulsive Δv applied at the start of each arc, in km/s
dv_kmps = [
    [0.0, 0.0,   0.0],
    [0.0, 0.005, 0.0],
    [0.0, 0.0,   0.0],
    [0.0, 0.0,  -0.005],
]

sols = []
for (k, dv) in enumerate(dv_kmps)
    # each arc starts where the previous one ended
    x_start = k == 1 ? copy(x0) : copy(sols[end].u[end])
    t_start = k == 1 ? 0.0 : sols[end].t[end]

    x_start[4:6] += dv / parameters.VU          # apply impulse
    prob = ODEProblem(eom_Nbody!, x_start, (t_start, t_start + arc_duration), parameters)
    push!(sols, solve(prob, Vern8(), reltol=1e-12, abstol=1e-12))
end
length(sols)

## Expected ODE solution format

The SPK writer expects each coast arc to behave like a standard `OrdinaryDiffEq.jl` solution:

```julia
sol.t       # nondimensional time grid
sol(t_nd)   # interpolated nondimensional state at time t_nd
```

Each state returned by `sol(t_nd)` should contain at least

```julia
[x, y, z, vx, vy, vz]
```

in nondimensional units. For example, for the first arc:

In [ ]:
t_nd = (sols[1].t[1] + sols[1].t[end]) / 2
sols[1](t_nd)

The parameter object must provide the canonical scaling values

In [ ]:
(
    TU = parameters.TU,   # seconds per nondimensional time unit
    DU = parameters.DU,   # kilometers per nondimensional distance unit
    VU = parameters.VU,   # kilometers/second per nondimensional velocity unit
)

Internally, states are converted before writing the SPK:

```julia
r_km   = x_nd[1:3] .* parameters.DU
v_kmps = x_nd[4:6] .* parameters.VU
et     = et0 + t_nd * parameters.TU
```

## Basic SPK generation

A minimal call looks like this (the generated files are written to a temporary directory; change `outdir` to keep them):

In [ ]:
outdir = mktempdir()
spice_id = -123456
center_id = 301

result = ode_sol_to_spk(
    sols,
    et0,
    parameters;
    output_spk = joinpath(outdir, "trajectory.bsp"),
    spice_id = spice_id,
    center_id = center_id,
    ref_frame_name = "J2000",
    dt_sec = 1800.0,
    segment_gap_sec = 1e-7,
    write_metadata = true,
)

Here:

- `sols` is a vector of ODE solution segments,
- `et0` is the reference epoch in seconds past J2000,
- `spice_id` is the NAIF object ID assigned to the generated trajectory,
- `center_id` is the NAIF ID of the central body,
- `ref_frame_name` is the frame used for the states written to the SPK,
- `dt_sec` is the sampling interval used for each SPK segment,
- `segment_gap_sec` trims the right endpoint of non-final segments to avoid overlapping SPK coverage at impulsive boundaries.

The returned `result` is a `NamedTuple` containing generated paths and summaries:

In [ ]:
for key in (:output_spk, :maneuver_txt, :ocp_maneuver_txt, :trajectory_maneuver_txt,
            :metadata_json, :segment_count, :epoch_ranges)
    println(rpad(key, 24), "=> ", getproperty(result, key))
end

## BSP/SPK file

The main output is the BSP file:

In [ ]:
result.output_spk

By default, `ode_sol_to_spk` writes one SPK segment per ODE solution segment. This is useful for impulsive trajectories because it avoids interpolating across discontinuous velocity jumps.

If several ODE solution segments should be grouped into one SPK segment, pass explicit coast windows. Each tuple in `coast_windows` gives the first and last ODE solution segment included in that SPK segment. Only group segments that are continuous; in our trajectory, the second and third arcs can be merged:

In [ ]:
result_windows = ode_sol_to_spk(
    sols,
    et0,
    parameters;
    output_spk = joinpath(outdir, "trajectory_windows.bsp"),
    spice_id = spice_id,
    center_id = center_id,
    coast_windows = [(1, 1), (2, 3), (4, 4)],
    write_metadata = false,
)
result_windows.segment_count

## Incremental append workflow

For station-keeping simulations, the function can append one propagated ODE arc at a time. This avoids storing every ODE solution in memory until the end of the run.

In the loop below, each `sol_recurse` stands in for the ODE solution of one recursion interval, which would typically be obtained by solving an OCP, applying the (possibly corrupted) first maneuver, and propagating the truth dynamics.

In [ ]:
output_spk = joinpath(outdir, "sk_seed_001_native_true.bsp")
N_recurse = length(sols)

for k in 1:N_recurse
    sol_recurse = sols[k]

    result_k = ode_sol_to_spk(
        sol_recurse,
        et0,
        parameters;
        output_spk = output_spk,
        spice_id = -200001,
        center_id = 301,
        ref_frame_name = "J2000",
        append = k > 1,
        overwrite = k == 1,
        segment_index = k,
        dt_sec = 1800.0,
        segment_gap_sec = k < N_recurse ? 1e-7 : 0.0,
        write_metadata = false,
        verbose = false,
        show_progress = false,
    )
end

The first call creates the BSP. Later calls append new SPK type-13 segments to the same BSP.

## Maneuver files

There are two maneuver-file concepts.

### Primary executed/OCP maneuver file

For optimal-control or station-keeping workflows, the main maneuver product can be written from a control history. Here, we build the control history from the impulses applied to our trajectory: column `k` of `ocp_control` holds the nondimensional Δv applied at the nondimensional time `ocp_control_times[k]`.

In [ ]:
idx_maneuvers = findall(dv -> norm(dv) > 0, dv_kmps)        # arcs starting with an impulse
ocp_control = hcat([dv_kmps[k] / parameters.VU for k in idx_maneuvers]...)
ocp_control_times = [sols[k].t[1] for k in idx_maneuvers]

result = ode_sol_to_spk(
    sols,
    et0,
    parameters;
    output_spk = joinpath(outdir, "trajectory.bsp"),
    spice_id = spice_id,
    center_id = center_id,
    ocp_control = ocp_control,
    ocp_control_times = ocp_control_times,
    write_ocp_maneuvers = true,
)
print(read(result.ocp_maneuver_txt, String))

Rows 1:3 of `ocp_control` are the vector control components. If row 4 is present, it is treated as the scalar control/slack variable used by the OCP formulation. The scalar convention is summarized in metadata.

For station-keeping truth products, pass the actually applied/corrupted control history as the `ocp_control` input. In that case, the maneuver file represents executed maneuvers, not the originally planned control.

### Trajectory-jump diagnostic maneuver file

If `write_maneuvers = true`, the helper can also write a diagnostic maneuver file reconstructed from velocity jumps between adjacent ODE solution segments:

In [ ]:
result = ode_sol_to_spk(
    sols,
    et0,
    parameters;
    output_spk = joinpath(outdir, "trajectory.bsp"),
    spice_id = spice_id,
    center_id = center_id,
    write_maneuvers = true,
)
print(read(result.trajectory_maneuver_txt, String))

The diagnostic file is separate from the executed/OCP control file. It contains:

```text
# ETSECONDS,DVX_mps,DVY_mps,DVZ_mps
```

The reconstructed velocity jumps match the 5 m/s impulses applied at the start of the second and fourth arcs, while the jump at the continuous boundary between the second and third arcs is at the level of numerical noise.

For incremental single-arc station-keeping output, trajectory-jump diagnostics are usually disabled because each call only has one arc.

## Metadata JSON

If `write_metadata = true`, the helper writes a metadata sidecar file:

In [ ]:
result.metadata_json

The metadata file records information such as:

- generated product paths,
- SPICE object ID or object name,
- center ID or center name,
- reference frame,
- SPK type and polynomial degree,
- optional leapseconds/frame-definition paths for traceability,
- segment coverage windows,
- sampling interval and segment gap,
- nondimensional scaling values,
- optional force-model metadata,
- maneuver summary information.

Force-model metadata can be supplied explicitly:

In [ ]:
result = ode_sol_to_spk(
    sols,
    et0,
    parameters;
    output_spk = joinpath(outdir, "trajectory.bsp"),
    spice_id = spice_id,
    center_id = center_id,
    force_model_metadata = Dict(
        "dynamics" => "eom_NbodySH!",
        "SRP" => Dict(
            "enabled" => true,
            "Cr" => 1.15,
            "A_over_m" => 0.016,
            "P0" => 4.56e-6,
        ),
        "spherical_harmonics" => Dict(
            "enabled" => true,
            "body" => "Moon",
            "degree" => 4,
            "body_fixed_frame" => "MOON_PA",
        ),
    ),
)

Alternatively, common entries can be supplied through keyword arguments. Here, we record the force model actually used to propagate `sols`:

In [ ]:
result = ode_sol_to_spk(
    sols,
    et0,
    parameters;
    output_spk = joinpath(outdir, "trajectory.bsp"),
    spice_id = spice_id,
    center_id = center_id,
    dynamics_name = "eom_Nbody!",
    srp_enabled = false,
    spherical_harmonics_enabled = false,
)
print(read(result.metadata_json, String))

If the trajectory had been propagated with `eom_NbodySH!` and SRP, the corresponding keyword arguments would be:

```julia
dynamics_name = "eom_NbodySH!",
srp_enabled = true,
srp_Cr = 1.15,
srp_Am = 0.016,
srp_P0 = 4.56e-6,
spherical_harmonics_enabled = true,
spherical_harmonics_body = "Moon",
spherical_harmonics_nmax = 4,
spherical_harmonics_frame = "MOON_PA",
```

## Label files

A label file is a lightweight sidecar file used by downstream visualization or analysis scripts to associate the generated BSP with an object name, SPICE ID, center, frame, and display settings.

The SPK writer returns the information needed to create such a label file:

```julia
result.output_spk
spice_id
center_id
ref_frame_name
```

The exact label format depends on the downstream tool. A simple label-writing step can therefore be kept separate from `ode_sol_to_spk`.

## Intermediate/debug files

The native pipeline samples states in memory and writes the BSP directly with SPICE. It does not need setup files or an external SPK-writing executable.

For debugging, state text files can still be written with lower-level helper functions such as `write_segmented_states_for_spk!` or `write_solution_segment_states_for_spk!`. These files are for inspection only; they are not required for normal BSP generation.

## Validating the generated BSP

A generated BSP can be validated by furnishing it with SPICE, querying states from the BSP, and comparing those states against the original ODE solution:

In [ ]:
furnsh(result.output_spk)

try
    for (k, sol) in enumerate(sols)
        # query an epoch in the middle of the k-th arc
        et_query = et0 + (sol.t[1] + sol.t[end]) / 2 * parameters.TU
        state_spk, _ = spkezr(
            string(spice_id),
            et_query,
            "J2000",
            "NONE",
            string(center_id),
        )

        t_nd = (et_query - et0) / parameters.TU
        x_ref = sol(t_nd)

        state_ref = vcat(
            x_ref[1:3] .* parameters.DU,
            x_ref[4:6] .* parameters.VU,
        )

        @assert isapprox(state_spk[1:3], state_ref[1:3]; atol = 1e-5)   # km
        @assert isapprox(state_spk[4:6], state_ref[4:6]; atol = 1e-8)   # km/s
        println("arc $k: position error = $(norm(state_spk[1:3] - state_ref[1:3])) km")
    end
finally
    unload(result.output_spk)
end

This is the same validation strategy used in the automated end-to-end tests. The tests generate small BSPs with native SPICE, furnish them, query states from the BSP, compare them against the original ODE solutions, unload the BSPs, and check that temporary files are cleaned up.

Finally, we remove the temporary output directory:

In [ ]:
rm(outdir; recursive = true)